In [0]:
WITH test_windows AS (
  SELECT *
  FROM VALUES
    ('test_1', TIMESTAMP '2026-04-05 22:00:00', TIMESTAMP '2026-04-06 00:00:00'),
    ('test_2', TIMESTAMP '2026-04-06 08:00:00', TIMESTAMP '2026-04-06 10:00:00'),
    ('test_3', TIMESTAMP '2026-04-06 10:00:00', TIMESTAMP '2026-04-06 12:00:00'),
    ('test_4', TIMESTAMP '2026-04-06 12:00:00', TIMESTAMP '2026-04-06 14:00:00'),
    ('test_5', TIMESTAMP '2026-04-06 14:00:00', TIMESTAMP '2026-04-06 16:00:00')
  AS test_windows(test_id, test_start, test_end)
),
usage_with_test AS (
  SELECT
    t.test_id,
    t.test_start,
    t.test_end,
    u.custom_tags,
    u.sku_name,
    u.cloud,
    u.usage_quantity,
    u.usage_start_time,
    u.usage_end_time,
    u.billing_origin_product,
    u.usage_metadata
  FROM system.billing.usage u
  JOIN test_windows t
    ON u.usage_start_time >= t.test_start
   AND u.usage_end_time   <= t.test_end
  WHERE
    map_contains_key(u.custom_tags, 'benchmark_test')
    OR map_contains_key(u.custom_tags, 'benchmark_policy')
)
SELECT
  u.test_id,
  CASE
    WHEN custom_tags['benchmark_test'] IS NOT NULL THEN custom_tags['benchmark_test']
    WHEN custom_tags['benchmark_policy'] IS NOT NULL THEN custom_tags['benchmark_policy']
  END AS benchmark_name,
  SUM(u.usage_quantity * lp.pricing.effective_list.default) AS price_usd
FROM usage_with_test u
LEFT JOIN system.billing.list_prices lp
  ON u.sku_name = lp.sku_name
 AND u.cloud = lp.cloud
 AND u.usage_start_time >= lp.price_start_time
 AND (u.usage_start_time < lp.price_end_time OR lp.price_end_time IS NULL)
GROUP BY
  u.test_id,
  benchmark_name
HAVING benchmark_name <> 'generate_source_data' AND benchmark_name <> 'cdf-benchmark-generate-source-data'
ORDER BY u.test_id, price_usd DESC
;
-- validate add original product JOB/PIPELINE and charge reson/type

Databricks visualization. Run in Databricks to view.